# (7) Transfer Learning for Motor Performance Prediction

This chapter explores transfer learning techniques for motor performance map prediction, focusing on how knowledge learned from one motor design or operating condition can be applied to improve performance on related tasks with limited data.

## Learning Objectives

- Understand the motivation for transfer learning in motor design applications
- Master internal transfer learning between different motor operating regimes
- Explore external transfer learning from simulation to real-world data
- Implement domain adaptation techniques for motor performance prediction
- Learn strategies for effective knowledge transfer in motor applications

## 7.1 Motivation for Transfer Learning in Motor Design

### 7.1.1 Data Scarcity Challenges

Motor performance prediction often faces significant data challenges:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("🚀 Transfer Learning for Motor Performance Prediction")
print("=" * 60)
print("🎯 Chapter Goals:")
print("  • Understand transfer learning fundamentals")
print("  • Implement internal transfer between operating regimes")
print("  • Apply external transfer from simulation to real-world")
print("  • Master domain adaptation techniques")

### 7.1.2 Types of Transfer Learning in Motor Applications

Transfer learning can be categorized based on the relationship between source and target domains:

1. **Internal Transfer Learning**: Same motor type, different operating conditions
2. **External Transfer Learning**: Different motor types or simulation-to-real
3. **Multi-task Learning**: Simultaneous learning of related motor performance tasks

In [ ]:
def transfer_learning_motivation_demo():
    """Demonstrate the motivation for transfer learning in motor applications"""
    
    # Simulate data availability scenarios
    motor_types = ['IPM-50kW', 'IPM-75kW', 'FSCW-50kW', 'FSCW-75kW']
    data_scenarios = {
        'High Fidelity Simulation': [10000, 10000, 10000, 10000],
        'Lab Testing': [500, 300, 400, 200],
        'Field Data': [100, 50, 80, 30]
    }
    
    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Transfer Learning Motivation in Motor Applications', fontsize=16, fontweight='bold')
    
    # Plot 1: Data availability across scenarios
    ax1 = axes[0, 0]
    x = np.arange(len(motor_types))
    width = 0.25
    
    for i, (scenario, data) in enumerate(data_scenarios.items()):
        ax1.bar(x + i * width, data, width, label=scenario, alpha=0.7)
    
    ax1.set_xlabel('Motor Type', fontweight='bold')
    ax1.set_ylabel('Number of Data Points', fontweight='bold')
    ax1.set_title('Data Availability Across Motor Types', fontweight='bold')
    ax1.set_xticks(x + width * 1.5)
    ax1.set_xticklabels(motor_types, rotation=45)
    ax1.legend()
    ax1.set_yscale('log')
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Plot 2: Performance vs data size
    ax2 = axes[0, 1]
    data_sizes = np.logspace(1, 4, 20)
    
    # Simulate performance curves
    from_scratch_perf = 0.85 * (1 - np.exp(-data_sizes / 1000))
    transfer_perf = 0.85 * (1 - np.exp(-data_sizes / 300)) + 0.1  # Better baseline
    
    ax2.loglog(data_sizes, from_scratch_perf, 'b-', linewidth=2, label='From Scratch')
    ax2.loglog(data_sizes, transfer_perf, 'r-', linewidth=2, label='With Transfer Learning')
    ax2.set_xlabel('Training Data Size', fontweight='bold')
    ax2.set_ylabel('Model Performance (Accuracy)', fontweight='bold')
    ax2.set_title('Learning Curves: Transfer vs From Scratch', fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Highlight data efficiency gain
    target_perf = 0.7
    from_scratch_size = data_sizes[np.where(from_scratch_perf >= target_perf)[0][0]]
    transfer_size = data_sizes[np.where(transfer_perf >= target_perf)[0][0]]
    
    ax2.axhline(y=target_perf, color='green', linestyle='--', alpha=0.7)
    ax2.axvline(x=from_scratch_size, color='blue', linestyle=':', alpha=0.7)
    ax2.axvline(x=transfer_size, color='red', linestyle=':', alpha=0.7)
    
    # Plot 3: Domain similarity visualization
    ax3 = axes[1, 0]
    
    # Create domain similarity matrix
    domains = ['IPM-50kW Sim', 'IPM-75kW Sim', 'FSCW-50kW Sim', 'FSCW-75kW Sim',
               'IPM-50kW Real', 'IPM-75kW Real']
    
    similarity_matrix = np.array([
        [1.0, 0.9, 0.6, 0.5, 0.8, 0.7],  # IPM-50kW Sim
        [0.9, 1.0, 0.5, 0.6, 0.7, 0.8],  # IPM-75kW Sim
        [0.6, 0.5, 1.0, 0.9, 0.5, 0.4],  # FSCW-50kW Sim
        [0.5, 0.6, 0.9, 1.0, 0.4, 0.5],  # FSCW-75kW Sim
        [0.8, 0.7, 0.5, 0.4, 1.0, 0.9],  # IPM-50kW Real
        [0.7, 0.8, 0.4, 0.5, 0.9, 1.0],  # IPM-75kW Real
    ])
    
    im = ax3.imshow(similarity_matrix, cmap='RdYlBu_r', aspect='auto', vmin=0, vmax=1)
    ax3.set_xticks(range(len(domains)))
    ax3.set_yticks(range(len(domains)))
    ax3.set_xticklabels(domains, rotation=45, ha='right')
    ax3.set_yticklabels(domains)
    ax3.set_title('Domain Similarity Matrix', fontweight='bold')
    
    # Add similarity values
    for i in range(len(domains)):
        for j in range(len(domains)):
            ax3.text(j, i, f'{similarity_matrix[i, j]:.1f}',
                    ha='center', va='center', color='black', fontweight='bold')
    
    plt.colorbar(im, ax=ax3, label='Similarity Score')
    
    # Plot 4: Transfer learning benefits summary
    ax4 = axes[1, 1]
    benefits = ['Reduced Training Time', 'Better Performance', 'Lower Data Requirements',
               'Faster Convergence', 'Improved Generalization']
    importance = [4.5, 4.8, 4.2, 4.0, 4.3]  # 1-5 scale
    
    bars = ax4.barh(benefits, importance, color='lightgreen', alpha=0.7)
    ax4.set_xlabel('Importance (1-5 scale)', fontweight='bold')
    ax4.set_title('Transfer Learning Benefits', fontweight='bold')
    ax4.set_xlim(0, 5)
    ax4.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for bar, value in zip(bars, importance):
        width = bar.get_width()
        ax4.text(width + 0.1, bar.get_y() + bar.get_height()/2,
                f'{value:.1f}', ha='left', va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print insights
    print("💡 Transfer Learning Insights:")
    print("=" * 40)
    print(f"📊 Data efficiency improvement: {from_scratch_size/transfer_size:.1f}x")
    print(f"🎯 Target performance ({target_perf}) achieved with {transfer_size:.0f} samples (vs {from_scratch_size:.0f})")
    print(f"🔗 Highest domain similarity: {np.max(similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]):.2f}")
    print("\n🎯 Key Benefits:")
    print("  • Leverages existing knowledge from similar motors")
    print("  • Reduces required training data significantly")
    print("  • Improves performance on data-scarce applications")
    print("  • Enables rapid deployment of new motor designs")
    
    return similarity_matrix

# Run motivation demonstration
similarity_matrix = transfer_learning_motivation_demo()

## 7.2 Internal Transfer Learning

Internal transfer learning involves transferring knowledge within the same motor type but across different operating conditions or power ratings.

In [ ]:
class MotorGRUTransfer(nn.Module):
    """GRU model with transfer learning capabilities for motor performance prediction"""
    
    def __init__(self, input_dim=4, hidden_dim=64, output_dim=3, num_layers=2, 
                 pretrained_layers=None):
        super().__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.num_layers = num_layers
        
        # Input projection
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        
        # GRU layers
        self.gru_layers = nn.ModuleList()
        for i in range(num_layers):
            self.gru_layers.append(
                nn.GRU(hidden_dim if i == 0 else hidden_dim, 
                      hidden_dim, batch_first=True)
            )
        
        # Output projection
        self.output_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim // 2, output_dim)
        )
        
        # Load pretrained weights if provided
        if pretrained_layers is not None:
            self._load_pretrained_layers(pretrained_layers)
    
    def _load_pretrained_layers(self, pretrained_dict):
        """Load pretrained weights into compatible layers"""
        model_dict = self.state_dict()
        
        # Filter and load compatible layers
        pretrained_dict = {k: v for k, v in pretrained_dict.items() 
                         if k in model_dict and v.size() == model_dict[k].size()}
        
        model_dict.update(pretrained_dict)
        self.load_state_dict(model_dict)
        
        print(f"Loaded {len(pretrained_dict)} pretrained parameters")
    
    def forward(self, x, hidden=None):
        batch_size, seq_len, _ = x.size()
        
        # Input projection
        x = self.input_proj(x)
        
        # Pass through GRU layers
        for gru_layer in self.gru_layers:
            x, hidden = gru_layer(x, hidden)
        
        # Output projection
        output = self.output_proj(x)
        
        return output, hidden
    
    def get_feature_extractor(self):
        """Return the feature extraction part of the model"""
        return nn.Sequential(
            self.input_proj,
            *self.gru_layers
        )

def generate_motor_data(num_samples=1000, motor_type='IPM', power_rating=50, 
                       noise_level=0.05, seed=None):
    """Generate synthetic motor performance data"""
    
    if seed is not None:
        np.random.seed(seed)
    
    # Generate operating conditions
    speeds = np.random.uniform(1000, 6000, num_samples)
    torques = np.random.uniform(20, 200, num_samples)
    currents = np.random.uniform(10, 150, num_samples)
    temperatures = np.random.uniform(20, 80, num_samples)
    
    # Motor-specific parameters
    if motor_type == 'IPM':
        efficiency_base = 0.92
        power_loss_factor = 0.08
        thermal_sensitivity = 0.5
    else:  # FSCW
        efficiency_base = 0.88
        power_loss_factor = 0.12
        thermal_sensitivity = 0.7
    
    # Power rating effects
    power_factor = power_rating / 50.0
    
    # Calculate performance metrics
    # Efficiency: decreases at high speeds and torques
    efficiency = efficiency_base - 0.1 * (speeds / 6000)**2 - 0.05 * (torques / 200)**2
    efficiency += thermal_sensitivity * (temperatures - 50) / 100
    efficiency = np.clip(efficiency, 0.6, 0.98)
    
    # Power loss: increases with current and temperature
    power_loss = power_loss_factor * (currents / 100)**2 * power_factor
    power_loss *= (1 + 0.01 * (temperatures - 50))
    
    # Thermal rise: based on power loss and cooling
    thermal_rise = power_loss * 50 * power_factor / (1 + 0.01 * speeds)
    
    # Add noise
    efficiency += np.random.normal(0, noise_level, num_samples)
    power_loss += np.random.normal(0, noise_level * 10, num_samples)
    thermal_rise += np.random.normal(0, noise_level * 5, num_samples)
    
    # Create input features
    input_features = np.column_stack([
        speeds, torques, currents, temperatures
    ])
    
    # Create output targets
    output_targets = np.column_stack([
        efficiency, power_loss, thermal_rise
    ])
    
    return input_features, output_targets

def create_sequences(data_X, data_y, seq_len=10):
    """Create sequences from time series data"""
    
    sequences_X = []
    sequences_y = []
    
    for i in range(len(data_X) - seq_len + 1):
        sequences_X.append(data_X[i:i+seq_len])
        sequences_y.append(data_y[i+seq_len-1])  # Predict last element
    
    return np.array(sequences_X), np.array(sequences_y)

# Generate source and target domain data
print("🔄 Generating Source and Target Domain Data...")

# Source domain: IPM 50kW with ample data
source_X, source_y = generate_motor_data(
    num_samples=2000, motor_type='IPM', power_rating=50, noise_level=0.03, seed=42
)

# Target domain: IPM 75kW with limited data
target_X, target_y = generate_motor_data(
    num_samples=200, motor_type='IPM', power_rating=75, noise_level=0.05, seed=123
)

print(f"✅ Source domain: {source_X.shape[0]} samples")
print(f"✅ Target domain: {target_X.shape[0]} samples")

In [ ]:
def demonstrate_internal_transfer_learning():
    """Demonstrate internal transfer learning between motor power ratings"""
    
    # Create sequences
    seq_len = 8
    source_X_seq, source_y_seq = create_sequences(source_X, source_y, seq_len)
    target_X_seq, target_y_seq = create_sequences(target_X, target_y, seq_len)
    
    # Normalize data
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    
    # Fit on source data
    source_X_scaled = scaler_X.fit_transform(source_X_seq.reshape(-1, source_X_seq.shape[-1]))
    source_y_scaled = scaler_y.fit_transform(source_y_seq)
    
    source_X_scaled = source_X_scaled.reshape(source_X_seq.shape)
    target_X_scaled = scaler_X.transform(target_X_seq.reshape(-1, target_X_seq.shape[-1]))
    target_X_scaled = target_X_scaled.reshape(target_X_seq.shape)
    target_y_scaled = scaler_y.transform(target_y_seq)
    
    # Convert to tensors
    source_X_tensor = torch.FloatTensor(source_X_scaled)
    source_y_tensor = torch.FloatTensor(source_y_scaled)
    target_X_tensor = torch.FloatTensor(target_X_scaled)
    target_y_tensor = torch.FloatTensor(target_y_scaled)
    
    # Create datasets
    source_dataset = TensorDataset(source_X_tensor, source_y_tensor)
    target_dataset = TensorDataset(target_X_tensor, target_y_tensor)
    
    source_loader = DataLoader(source_dataset, batch_size=32, shuffle=True)
    target_loader = DataLoader(target_dataset, batch_size=16, shuffle=True)
    
    # Training scenarios
    scenarios = {
        'From Scratch': None,
        'Full Transfer': 'full',
        'Partial Transfer': 'partial'
    }
    
    results = {}
    
    for scenario_name, transfer_type in scenarios.items():
        print(f"\n🚀 Training: {scenario_name}")
        
        # Initialize model
        if transfer_type == 'full' or transfer_type == 'partial':
            # First train on source domain
            source_model = MotorGRUTransfer(input_dim=4, hidden_dim=64, output_dim=3)
            
            # Train on source domain
            optimizer = torch.optim.Adam(source_model.parameters(), lr=0.001)
            criterion = nn.MSELoss()
            
            for epoch in range(20):  # Fewer epochs for source training
                source_model.train()
                total_loss = 0
                
                for batch_X, batch_y in source_loader:
                    optimizer.zero_grad()
                    outputs, _ = source_model(batch_X)
                    loss = criterion(outputs, batch_y)
                    loss.backward()
                    optimizer.step()
                    total_loss += loss.item()
                
                if epoch % 5 == 0:
                    print(f"  Source Epoch {epoch}: Loss = {total_loss/len(source_loader):.4f}")
            
            # Create target model with transfer learning
            if transfer_type == 'full':
                # Full transfer: use all pretrained weights
                pretrained_dict = source_model.state_dict()
                target_model = MotorGRUTransfer(input_dim=4, hidden_dim=64, output_dim=3,
                                            pretrained_layers=pretrained_dict)
            else:  # partial transfer
                # Partial transfer: only transfer feature extractor
                pretrained_dict = {k: v for k, v in source_model.state_dict().items() 
                                 if 'output_proj' not in k}
                target_model = MotorGRUTransfer(input_dim=4, hidden_dim=64, output_dim=3,
                                            pretrained_layers=pretrained_dict)
        else:
            # Train from scratch
            target_model = MotorGRUTransfer(input_dim=4, hidden_dim=64, output_dim=3)
        
        # Fine-tune on target domain
        optimizer = torch.optim.Adam(target_model.parameters(), lr=0.001)
        criterion = nn.MSELoss()
        
        train_losses = []
        
        for epoch in range(50):
            target_model.train()
            total_loss = 0
            
            for batch_X, batch_y in target_loader:
                optimizer.zero_grad()
                outputs, _ = target_model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            
            avg_loss = total_loss / len(target_loader)
            train_losses.append(avg_loss)
            
            if epoch % 10 == 0:
                print(f"  Target Epoch {epoch}: Loss = {avg_loss:.4f}")
        
        # Evaluate on target domain
        target_model.eval()
        predictions = []
        actuals = []
        
        with torch.no_grad():
            for batch_X, batch_y in target_loader:
                outputs, _ = target_model(batch_X)
                predictions.append(outputs.numpy())
                actuals.append(batch_y.numpy())
        
        predictions = np.vstack(predictions)
        actuals = np.vstack(actuals)
        
        # Calculate metrics
        mse = mean_squared_error(actuals, predictions)
        mae = mean_absolute_error(actuals, predictions)
        
        # Convert back to original scale for interpretation
        predictions_orig = scaler_y.inverse_transform(predictions)
        actuals_orig = scaler_y.inverse_transform(actuals)
        
        results[scenario_name] = {
            'train_losses': train_losses,
            'mse': mse,
            'mae': mae,
            'predictions': predictions_orig,
            'actuals': actuals_orig
        }
        
        print(f"  ✅ Final MSE: {mse:.4f}, MAE: {mae:.4f}")
    
    # Visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Internal Transfer Learning: IPM 50kW → IPM 75kW', fontsize=16, fontweight='bold')
    
    # Plot 1: Training curves comparison
    ax1 = axes[0, 0]
    for scenario_name, result in results.items():
        ax1.plot(result['train_losses'], label=scenario_name, linewidth=2)
    
    ax1.set_xlabel('Epoch', fontweight='bold')
    ax1.set_ylabel('Training Loss', fontweight='bold')
    ax1.set_title('Training Convergence Comparison', fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_yscale('log')
    
    # Plot 2-4: Performance predictions for each metric
    metrics = ['Efficiency', 'Power Loss', 'Thermal Rise']
    colors = ['b', 'r', 'g']
    
    for i, (metric, color) in enumerate(zip(metrics, colors)):
        ax = axes[0, i + 1]
        
        for scenario_name, result in results.items():
            predictions = result['predictions'][:, i]
            actuals = result['actuals'][:, i]
            
            ax.scatter(actuals, predictions, alpha=0.6, label=scenario_name, s=20)
            
            # Perfect prediction line
            min_val = min(np.min(actuals), np.min(predictions))
            max_val = max(np.max(actuals), np.max(predictions))
            ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, label='Perfect')
        
        ax.set_xlabel(f'Actual {metric}', fontweight='bold')
        ax.set_ylabel(f'Predicted {metric}', fontweight='bold')
        ax.set_title(f'{metric} Prediction', fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    # Plot 5: Performance metrics comparison
    ax5 = axes[1, 0]
    scenario_names = list(results.keys())
    mse_values = [results[name]['mse'] for name in scenario_names]
    mae_values = [results[name]['mae'] for name in scenario_names]
    
    x = np.arange(len(scenario_names))
    width = 0.35
    
    bars1 = ax5.bar(x - width/2, mse_values, width, label='MSE', color='lightblue', alpha=0.7)
    bars2 = ax5.bar(x + width/2, mae_values, width, label='MAE', color='lightcoral', alpha=0.7)
    
    ax5.set_xlabel('Training Scenario', fontweight='bold')
    ax5.set_ylabel('Error (scaled)', fontweight='bold')
    ax5.set_title('Performance Metrics Comparison', fontweight='bold')
    ax5.set_xticks(x)
    ax5.set_xticklabels(scenario_names)
    ax5.legend()
    ax5.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax5.text(bar.get_x() + bar.get_width()/2., height + max(mse_values + mae_values)*0.01,
                    f'{height:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=9)
    
    # Plot 6: Sample time series predictions
    ax6 = axes[1, 1]
    sample_idx = 0
    time_steps = np.arange(seq_len)
    
    # Show actual vs predicted for efficiency
    for scenario_name, result in results.items():
        if scenario_name == 'Full Transfer':  # Show best performing model
            sample_prediction = result['predictions'][sample_idx:sample_idx+seq_len, 0]
            sample_actual = result['actuals'][sample_idx:sample_idx+seq_len, 0]
            
            ax6.plot(time_steps, sample_actual, 'o-', label='Actual', linewidth=2, markersize=6)
            ax6.plot(time_steps, sample_prediction, 's--', label='Predicted', linewidth=2, markersize=6)
            break
    
    ax6.set_xlabel('Time Step', fontweight='bold')
    ax6.set_ylabel('Efficiency', fontweight='bold')
    ax6.set_title('Sample Time Series Prediction', fontweight='bold')
    ax6.legend()
    ax6.grid(True, alpha=0.3)
    
    # Remove empty subplot
    axes[1, 2].remove()
    
    plt.tight_layout()
    plt.show()
    
    # Print results summary
    print("\n📊 Internal Transfer Learning Results:")
    print("=" * 50)
    for scenario_name, result in results.items():
        print(f"\n🎯 {scenario_name}:")
        print(f"  MSE: {result['mse']:.4f}")
        print(f"  MAE: {result['mae']:.4f}")
        print(f"  Final Training Loss: {result['train_losses'][-1]:.4f}")
    
    # Calculate improvement
    scratch_mse = results['From Scratch']['mse']
    full_transfer_mse = results['Full Transfer']['mse']
    partial_transfer_mse = results['Partial Transfer']['mse']
    
    print(f"\n🚀 Performance Improvements:")
    print(f"  Full Transfer vs From Scratch: {(1 - full_transfer_mse/scratch_mse)*100:.1f}% better")
    print(f"  Partial Transfer vs From Scratch: {(1 - partial_transfer_mse/scratch_mse)*100:.1f}% better")
    
    return results

# Run internal transfer learning demonstration
transfer_results = demonstrate_internal_transfer_learning()

## 7.3 External Transfer Learning

External transfer learning involves transferring knowledge across different motor types or from simulation to real-world data, which typically requires domain adaptation techniques.

In [ ]:
class DomainAdversarialNetwork(nn.Module):
    """Domain adversarial network for simulation-to-real transfer"""
    
    def __init__(self, feature_extractor, feature_dim=64, num_domains=2):
        super().__init__()
        
        self.feature_extractor = feature_extractor
        self.feature_dim = feature_dim
        
        # Domain classifier
        self.domain_classifier = nn.Sequential(
            nn.Linear(feature_dim, feature_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(feature_dim // 2, num_domains)
        )
        
        # Performance predictor (shared across domains)
        self.performance_predictor = nn.Sequential(
            nn.Linear(feature_dim, feature_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(feature_dim // 2, 3)  # efficiency, power_loss, thermal_rise
        )
        
        # Gradient reversal layer
        self.gradient_reversal = GradientReversalLayer()
    
    def forward(self, x, alpha=1.0):
        # Extract features
        features = self.feature_extractor(x)
        
        # Get last hidden state for each sequence
        if isinstance(features, tuple):  # GRU returns (output, hidden)
            features = features[0]  # Take output
        
        # Use last time step
        if len(features.shape) == 3:
            features = features[:, -1, :]  # (batch_size, feature_dim)
        
        # Performance prediction (forward gradient)
        performance_pred = self.performance_predictor(features)
        
        # Domain classification (reverse gradient)
        reversed_features = self.gradient_reversal(features, alpha)
        domain_pred = self.domain_classifier(reversed_features)
        
        return performance_pred, domain_pred, features

class GradientReversalLayer(nn.Module):
    """Gradient reversal layer for domain adversarial training"""
    
    def __init__(self):
        super().__init__()
    
    def forward(self, x, alpha):
        return GradientReversalFunction.apply(x, alpha)

class GradientReversalFunction(torch.autograd.Function):
    """Gradient reversal function"""
    
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None

def demonstrate_external_transfer_learning():
    """Demonstrate simulation-to-real transfer learning with domain adaptation"""
    
    # Generate simulation data (source domain)
    print("🔄 Generating Simulation and Real-world Data...")
    
    sim_X, sim_y = generate_motor_data(
        num_samples=2000, motor_type='IPM', power_rating=50, 
        noise_level=0.01, seed=42  # Low noise for simulation
    )
    
    # Generate real-world data (target domain) with systematic differences
    real_X, real_y = generate_motor_data(
        num_samples=300, motor_type='IPM', power_rating=50, 
        noise_level=0.08, seed=123  # Higher noise for real data
    )
    
    # Add systematic domain shift
    real_y[:, 0] -= 0.05  # Efficiency degradation in real world
    real_y[:, 1] *= 1.15   # Higher power losses
    real_y[:, 2] *= 1.10   # Higher thermal rise
    
    print(f"✅ Simulation data: {sim_X.shape[0]} samples")
    print(f"✅ Real-world data: {real_X.shape[0]} samples")
    
    # Create sequences
    seq_len = 8
    sim_X_seq, sim_y_seq = create_sequences(sim_X, sim_y, seq_len)
    real_X_seq, real_y_seq = create_sequences(real_X, real_y, seq_len)
    
    # Normalize data
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    
    # Fit on simulation data
    sim_X_scaled = scaler_X.fit_transform(sim_X_seq.reshape(-1, sim_X_seq.shape[-1]))
    sim_y_scaled = scaler_y.fit_transform(sim_y_seq)
    
    sim_X_scaled = sim_X_scaled.reshape(sim_X_seq.shape)
    real_X_scaled = scaler_X.transform(real_X_seq.reshape(-1, real_X_seq.shape[-1]))
    real_X_scaled = real_X_scaled.reshape(real_X_seq.shape)
    real_y_scaled = scaler_y.transform(real_y_seq)
    
    # Convert to tensors
    sim_X_tensor = torch.FloatTensor(sim_X_scaled)
    sim_y_tensor = torch.FloatTensor(sim_y_scaled)
    real_X_tensor = torch.FloatTensor(real_X_scaled)
    real_y_tensor = torch.FloatTensor(real_y_scaled)
    
    # Create domain labels
    sim_domain_labels = torch.zeros(len(sim_X_tensor), dtype=torch.long)  # Domain 0: simulation
    real_domain_labels = torch.ones(len(real_X_tensor), dtype=torch.long)   # Domain 1: real
    
    # Training strategies
    strategies = {
        'No Transfer': 'none',
        'Simple Transfer': 'simple',
        'Domain Adversarial': 'dann'
    }
    
    results = {}
    
    for strategy_name, strategy in strategies.items():
        print(f"\n🚀 Training: {strategy_name}")
        
        if strategy == 'dann':
            # Domain adversarial training
            feature_extractor = nn.Sequential(
                nn.Linear(4, 64),
                nn.ReLU(),
                nn.GRU(64, 64, batch_first=True)
            )
            
            model = DomainAdversarialNetwork(feature_extractor, feature_dim=64)
            
            # Optimizers
            optimizer_perf = torch.optim.Adam(
                list(model.feature_extractor.parameters()) + 
                list(model.performance_predictor.parameters()), 
                lr=0.001
            )
            optimizer_domain = torch.optim.Adam(
                model.domain_classifier.parameters(), 
                lr=0.001
            )
            
            criterion_perf = nn.MSELoss()
            criterion_domain = nn.CrossEntropyLoss()
            
            # Training loop with domain adversarial training
            for epoch in range(50):
                # Gradually increase alpha (domain adversarial strength)
                alpha = 2. / (1. + np.exp(-10 * epoch / 50)) - 1
                
                # Train on simulation data
                model.train()
                
                # Shuffle indices
                sim_indices = torch.randperm(len(sim_X_tensor))
                real_indices = torch.randperm(len(real_X_tensor))
                
                # Mini-batches
                batch_size = 32
                n_sim_batches = len(sim_X_tensor) // batch_size
                n_real_batches = min(len(real_X_tensor) // batch_size, 5)  # Limit real batches
                
                total_perf_loss = 0
                total_domain_loss = 0
                
                for i in range(max(n_sim_batches, n_real_batches)):
                    # Simulation batch
                    if i < n_sim_batches:
                        sim_batch_idx = sim_indices[i*batch_size:(i+1)*batch_size]
                        sim_batch_X = sim_X_tensor[sim_batch_idx]
                        sim_batch_y = sim_y_tensor[sim_batch_idx]
                        sim_batch_domain = sim_domain_labels[sim_batch_idx]
                        
                        # Performance prediction
                        optimizer_perf.zero_grad()
                        perf_pred, domain_pred, _ = model(sim_batch_X, alpha)
                        perf_loss = criterion_perf(perf_pred, sim_batch_y)
                        perf_loss.backward()
                        optimizer_perf.step()
                        
                        total_perf_loss += perf_loss.item()
                    
                    # Real batch for domain classification
                    if i < n_real_batches:
                        real_batch_idx = real_indices[i*batch_size:(i+1)*batch_size]
                        real_batch_X = real_X_tensor[real_batch_idx]
                        real_batch_domain = real_domain_labels[real_batch_idx]
                        
                        # Domain classification
                        optimizer_domain.zero_grad()
                        _, domain_pred_real, _ = model(real_batch_X, alpha)
                        
                        # Combine simulation and real for domain loss
                        if i < n_sim_batches:
                            combined_pred = torch.cat([domain_pred, domain_pred_real], dim=0)
                            combined_labels = torch.cat([sim_batch_domain, real_batch_domain], dim=0)
                        else:
                            combined_pred = domain_pred_real
                            combined_labels = real_batch_domain
                        
                        domain_loss = criterion_domain(combined_pred, combined_labels)
                        domain_loss.backward()
                        optimizer_domain.step()
                        
                        total_domain_loss += domain_loss.item()
                
                if epoch % 10 == 0:
                    print(f"  Epoch {epoch}: Perf Loss = {total_perf_loss/max(n_sim_batches,1):.4f}, "
                          f"Domain Loss = {total_domain_loss/max(n_real_batches,1):.4f}, Alpha = {alpha:.3f}")
            
        elif strategy == 'simple':
            # Simple transfer: pretrain on simulation, fine-tune on real
            # Feature extractor
            feature_extractor = nn.Sequential(
                nn.Linear(4, 64),
                nn.ReLU(),
                nn.GRU(64, 64, batch_first=True)
            )
            
            # Performance predictor
            performance_predictor = nn.Sequential(
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(32, 3)
            )
            
            # Pretrain on simulation
            optimizer = torch.optim.Adam(
                list(feature_extractor.parameters()) + list(performance_predictor.parameters()),
                lr=0.001
            )
            criterion = nn.MSELoss()
            
            for epoch in range(30):
                # Train on simulation
                indices = torch.randperm(len(sim_X_tensor))
                for i in range(0, len(sim_X_tensor), batch_size):
                    batch_idx = indices[i:i+batch_size]
                    batch_X = sim_X_tensor[batch_idx]
                    batch_y = sim_y_tensor[batch_idx]
                    
                    optimizer.zero_grad()
                    features, hidden = feature_extractor(batch_X)
                    if isinstance(features, tuple):
                        features = features[0]
                    last_features = features[:, -1, :]
                    pred = performance_predictor(last_features)
                    loss = criterion(pred, batch_y)
                    loss.backward()
                    optimizer.step()
                
                if epoch % 10 == 0:
                    print(f"  Pretrain Epoch {epoch}: Loss = {loss.item():.4f}")
            
            # Fine-tune on real data
            for epoch in range(20):
                indices = torch.randperm(len(real_X_tensor))
                for i in range(0, len(real_X_tensor), batch_size):
                    batch_idx = indices[i:i+batch_size]
                    batch_X = real_X_tensor[batch_idx]
                    batch_y = real_y_tensor[batch_idx]
                    
                    optimizer.zero_grad()
                    features, hidden = feature_extractor(batch_X)
                    if isinstance(features, tuple):
                        features = features[0]
                    last_features = features[:, -1, :]
                    pred = performance_predictor(last_features)
                    loss = criterion(pred, batch_y)
                    loss.backward()
                    optimizer.step()
                
                if epoch % 5 == 0:
                    print(f"  Finetune Epoch {epoch}: Loss = {loss.item():.4f}")
            
            # Wrap for evaluation
            model = lambda x, alpha=1.0: (
                performance_predictor(feature_extractor(x)[0][:, -1, :]), 
                None, 
                feature_extractor(x)[0][:, -1, :]
            )
            
        else:  # no transfer
            # Train only on real data
            feature_extractor = nn.Sequential(
                nn.Linear(4, 64),
                nn.ReLU(),
                nn.GRU(64, 64, batch_first=True)
            )
            
            performance_predictor = nn.Sequential(
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(32, 3)
            )
            
            optimizer = torch.optim.Adam(
                list(feature_extractor.parameters()) + list(performance_predictor.parameters()),
                lr=0.001
            )
            criterion = nn.MSELoss()
            
            for epoch in range(50):
                indices = torch.randperm(len(real_X_tensor))
                for i in range(0, len(real_X_tensor), batch_size):
                    batch_idx = indices[i:i+batch_size]
                    batch_X = real_X_tensor[batch_idx]
                    batch_y = real_y_tensor[batch_idx]
                    
                    optimizer.zero_grad()
                    features, hidden = feature_extractor(batch_X)
                    if isinstance(features, tuple):
                        features = features[0]
                    last_features = features[:, -1, :]
                    pred = performance_predictor(last_features)
                    loss = criterion(pred, batch_y)
                    loss.backward()
                    optimizer.step()
                
                if epoch % 10 == 0:
                    print(f"  Epoch {epoch}: Loss = {loss.item():.4f}")
            
            # Wrap for evaluation
            model = lambda x, alpha=1.0: (
                performance_predictor(feature_extractor(x)[0][:, -1, :]), 
                None, 
                feature_extractor(x)[0][:, -1, :]
            )
        
        # Evaluate on real data
        model.eval()
        predictions = []
        actuals = []
        features_list = []
        
        with torch.no_grad():
            for i in range(0, len(real_X_tensor), batch_size):
                batch_X = real_X_tensor[i:i+batch_size]
                batch_y = real_y_tensor[i:i+batch_size]
                
                pred, _, features = model(batch_X)
                predictions.append(pred.numpy())
                actuals.append(batch_y.numpy())
                features_list.append(features.numpy())
        
        predictions = np.vstack(predictions)
        actuals = np.vstack(actuals)
        
        # Calculate metrics
        mse = mean_squared_error(actuals, predictions)
        mae = mean_absolute_error(actuals, predictions)
        
        # Convert back to original scale
        predictions_orig = scaler_y.inverse_transform(predictions)
        actuals_orig = scaler_y.inverse_transform(actuals)
        
        results[strategy_name] = {
            'mse': mse,
            'mae': mae,
            'predictions': predictions_orig,
            'actuals': actuals_orig,
            'features': np.vstack(features_list)
        }
        
        print(f"  ✅ Final MSE: {mse:.4f}, MAE: {mae:.4f}")
    
    # Visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('External Transfer Learning: Simulation → Real World', fontsize=16, fontweight='bold')
    
    # Plot 1: Domain distribution comparison
    ax1 = axes[0, 0]
    if 'Domain Adversarial' in results:
        features_dann = results['Domain Adversarial']['features']
        # Use first two dimensions for visualization
        ax1.scatter(features_dann[:len(real_X_tensor)//2, 0], 
                   features_dann[:len(real_X_tensor)//2, 1], 
                   alpha=0.6, label='Real Data', s=20)
        ax1.set_xlabel('Feature 1', fontweight='bold')
        ax1.set_ylabel('Feature 2', fontweight='bold')
        ax1.set_title('Feature Distribution (DANN)', fontweight='bold')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
    
    # Plot 2-4: Performance predictions
    metrics = ['Efficiency', 'Power Loss', 'Thermal Rise']
    colors = ['b', 'r', 'g']
    
    for i, (metric, color) in enumerate(zip(metrics, colors)):
        ax = axes[0, i + 1]
        
        for strategy_name, result in results.items():
            predictions = result['predictions'][:, i]
            actuals = result['actuals'][:, i]
            
            ax.scatter(actuals, predictions, alpha=0.6, label=strategy_name, s=20)
        
        # Perfect prediction line
        all_values = np.concatenate([result['predictions'][:, i] for result in results.values()] +
                                   [result['actuals'][:, i] for result in results.values()])
        min_val, max_val = np.min(all_values), np.max(all_values)
        ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, label='Perfect')
        
        ax.set_xlabel(f'Actual {metric}', fontweight='bold')
        ax.set_ylabel(f'Predicted {metric}', fontweight='bold')
        ax.set_title(f'{metric} Prediction', fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    # Plot 5: Performance metrics comparison
    ax5 = axes[1, 0]
    strategy_names = list(results.keys())
    mse_values = [results[name]['mse'] for name in strategy_names]
    mae_values = [results[name]['mae'] for name in strategy_names]
    
    x = np.arange(len(strategy_names))
    width = 0.35
    
    bars1 = ax5.bar(x - width/2, mse_values, width, label='MSE', color='lightblue', alpha=0.7)
    bars2 = ax5.bar(x + width/2, mae_values, width, label='MAE', color='lightcoral', alpha=0.7)
    
    ax5.set_xlabel('Transfer Strategy', fontweight='bold')
    ax5.set_ylabel('Error (scaled)', fontweight='bold')
    ax5.set_title('Performance Comparison', fontweight='bold')
    ax5.set_xticks(x)
    ax5.set_xticklabels(strategy_names, rotation=45)
    ax5.legend()
    ax5.grid(True, alpha=0.3, axis='y')
    
    # Plot 6: Error distribution analysis
    ax6 = axes[1, 1]
    
    for strategy_name, result in results.items():
        errors = (result['predictions'] - result['actuals']).flatten()
        ax6.hist(errors, bins=20, alpha=0.5, label=strategy_name, density=True)
    
    ax6.set_xlabel('Prediction Error', fontweight='bold')
    ax6.set_ylabel('Density', fontweight='bold')
    ax6.set_title('Error Distribution Comparison', fontweight='bold')
    ax6.axvline(x=0, color='black', linestyle='--', alpha=0.5)
    ax6.legend()
    ax6.grid(True, alpha=0.3)
    
    # Remove empty subplot
    axes[1, 2].remove()
    
    plt.tight_layout()
    plt.show()
    
    # Print results summary
    print("\n📊 External Transfer Learning Results:")
    print("=" * 50)
    for strategy_name, result in results.items():
        print(f"\n🎯 {strategy_name}:")
        print(f"  MSE: {result['mse']:.4f}")
        print(f"  MAE: {result['mae']:.4f}")
    
    # Calculate improvements
    if 'No Transfer' in results and 'Domain Adversarial' in results:
        no_transfer_mse = results['No Transfer']['mse']
        dann_mse = results['Domain Adversarial']['mse']
        improvement = (1 - dann_mse/no_transfer_mse) * 100
        print(f"\n🚀 DANN vs No Transfer: {improvement:.1f}% improvement")
    
    return results

# Run external transfer learning demonstration
external_results = demonstrate_external_transfer_learning()

## 7.4 Multi-Task Learning

Multi-task learning involves simultaneously learning related motor performance tasks, which can improve generalization and data efficiency through shared representations.

In [ ]:
class MultiTaskMotorModel(nn.Module):
    """Multi-task learning model for motor performance prediction"""
    
    def __init__(self, input_dim=4, shared_dim=64, task_dims=None):
        super().__init__()
        
        if task_dims is None:
            task_dims = {'efficiency': 1, 'power_loss': 1, 'thermal_rise': 1, 
                       'torque_ripple': 1, 'vibration': 1}
        
        self.shared_layers = nn.Sequential(
            nn.Linear(input_dim, shared_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(shared_dim, shared_dim),
            nn.ReLU()
        )
        
        # Task-specific layers
        self.task_layers = nn.ModuleDict({
            task: nn.Sequential(
                nn.Linear(shared_dim, shared_dim // 2),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(shared_dim // 2, task_dim)
            ) for task, task_dim in task_dims.items()
        })
        
        self.tasks = list(task_dims.keys())
    
    def forward(self, x, task=None):
        shared_features = self.shared_layers(x)
        
        if task is not None:
            return self.task_layers[task](shared_features)
        else:
            return {task: self.task_layers[task](shared_features) for task in self.tasks}
    
    def get_shared_features(self, x):
        return self.shared_layers(x)

def demonstrate_multi_task_learning():
    """Demonstrate multi-task learning for motor performance prediction"""
    
    # Generate multi-task motor data
    print("🔄 Generating Multi-Task Motor Data...")
    
    np.random.seed(42)
    num_samples = 1500
    
    # Operating conditions
    speeds = np.random.uniform(1000, 6000, num_samples)
    torques = np.random.uniform(20, 200, num_samples)
    currents = np.random.uniform(10, 150, num_samples)
    temperatures = np.random.uniform(20, 80, num_samples)
    
    # Multiple performance metrics
    efficiency = 0.9 - 0.15 * (speeds / 6000)**2 - 0.1 * (torques / 200)**2
    efficiency += np.random.normal(0, 0.03, num_samples)
    efficiency = np.clip(efficiency, 0.6, 0.98)
    
    power_loss = 0.1 * (currents / 100)**2 * (1 + 0.01 * (temperatures - 50))
    power_loss += np.random.normal(0, 0.01, num_samples)
    
    thermal_rise = power_loss * 40 / (1 + 0.01 * speeds)
    thermal_rise += np.random.normal(0, 2, num_samples)
    
    # Additional tasks
    torque_ripple = 0.05 * (torques / 200) * (1 + 0.1 * np.sin(speeds / 1000))
    torque_ripple += np.random.normal(0, 0.005, num_samples)
    
    vibration = 0.02 * (speeds / 6000) * (1 + 0.05 * currents / 100)
    vibration += np.random.normal(0, 0.002, num_samples)
    
    # Input features
    X = np.column_stack([speeds, torques, currents, temperatures])
    
    # Multi-task outputs
    y_tasks = {
        'efficiency': efficiency.reshape(-1, 1),
        'power_loss': power_loss.reshape(-1, 1),
        'thermal_rise': thermal_rise.reshape(-1, 1),
        'torque_ripple': torque_ripple.reshape(-1, 1),
        'vibration': vibration.reshape(-1, 1)
    }
    
    print(f"✅ Generated {num_samples} samples for {len(y_tasks)} tasks")
    
    # Training strategies
    strategies = {
        'Single Task (Efficiency)': 'single',
        'Independent Multi-Task': 'independent',
        'Shared Multi-Task': 'shared'
    }
    
    results = {}
    
    for strategy_name, strategy in strategies.items():
        print(f"\n🚀 Training: {strategy_name}")
        
        if strategy == 'single':
            # Single task learning on efficiency only
            model = MultiTaskMotorModel(input_dim=4, shared_dim=64, 
                                     task_dims={'efficiency': 1})
            
            optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
            criterion = nn.MSELoss()
            
            X_tensor = torch.FloatTensor(X)
            y_tensor = torch.FloatTensor(y_tasks['efficiency'])
            
            for epoch in range(100):
                model.train()
                optimizer.zero_grad()
                predictions = model(X_tensor, 'efficiency')
                loss = criterion(predictions, y_tensor)
                loss.backward()
                optimizer.step()
                
                if epoch % 20 == 0:
                    print(f"  Epoch {epoch}: Loss = {loss.item():.4f}")
            
            # Evaluate on all tasks (to show generalization)
            model.eval()
            single_results = {}
            with torch.no_grad():
                for task_name, y_values in y_tasks.items():
                    if task_name == 'efficiency':
                        pred = model(X_tensor, 'efficiency').numpy()
                    else:
                        # Use shared features but no task-specific training
                        shared_features = model.get_shared_features(X_tensor)
                        # Simple linear projection for evaluation
                        if task_name == 'power_loss':
                            pred = (shared_features[:, 0] * 0.1).numpy().reshape(-1, 1)
                        elif task_name == 'thermal_rise':
                            pred = (shared_features[:, 1] * 5).numpy().reshape(-1, 1)
                        else:
                            pred = np.zeros((len(X), 1))
                    
                    mse = mean_squared_error(y_values, pred)
                    single_results[task_name] = {'predictions': pred, 'mse': mse}
            
            results[strategy_name] = single_results
            
        elif strategy == 'independent':
            # Independent models for each task
            independent_results = {}
            
            for task_name, y_values in y_tasks.items():
                task_model = MultiTaskMotorModel(input_dim=4, shared_dim=64,
                                               task_dims={task_name: 1})
                
                optimizer = torch.optim.Adam(task_model.parameters(), lr=0.001)
                criterion = nn.MSELoss()
                
                X_tensor = torch.FloatTensor(X)
                y_tensor = torch.FloatTensor(y_values)
                
                for epoch in range(50):  # Fewer epochs for independent training
                    task_model.train()
                    optimizer.zero_grad()
                    predictions = task_model(X_tensor, task_name)
                    loss = criterion(predictions, y_tensor)
                    loss.backward()
                    optimizer.step()
                
                # Evaluate
                task_model.eval()
                with torch.no_grad():
                    pred = task_model(X_tensor, task_name).numpy()
                    mse = mean_squared_error(y_values, pred)
                    independent_results[task_name] = {'predictions': pred, 'mse': mse}
            
            results[strategy_name] = independent_results
            
        else:  # shared multi-task
            # Multi-task learning with shared representations
            model = MultiTaskMotorModel(input_dim=4, shared_dim=64)
            
            optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
            criterion = nn.MSELoss()
            
            X_tensor = torch.FloatTensor(X)
            
            for epoch in range(100):
                model.train()
                optimizer.zero_grad()
                
                # Forward pass for all tasks
                predictions = model(X_tensor)
                
                # Compute loss for all tasks
                total_loss = 0
                for task_name in model.tasks:
                    y_tensor = torch.FloatTensor(y_tasks[task_name])
                    task_loss = criterion(predictions[task_name], y_tensor)
                    total_loss += task_loss
                
                total_loss = total_loss / len(model.tasks)  # Average loss
                total_loss.backward()
                optimizer.step()
                
                if epoch % 20 == 0:
                    print(f"  Epoch {epoch}: Loss = {total_loss.item():.4f}")
            
            # Evaluate all tasks
            model.eval()
            shared_results = {}
            with torch.no_grad():
                predictions = model(X_tensor)
                for task_name in model.tasks:
                    pred = predictions[task_name].numpy()
                    mse = mean_squared_error(y_tasks[task_name], pred)
                    shared_results[task_name] = {'predictions': pred, 'mse': mse}
            
            results[strategy_name] = shared_results
    
    # Visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Multi-Task Learning for Motor Performance Prediction', fontsize=16, fontweight='bold')
    
    # Plot 1: Performance comparison across tasks
    ax1 = axes[0, 0]
    task_names = list(y_tasks.keys())
    
    for strategy_name in strategies.keys():
        mse_values = [results[strategy_name][task]['mse'] for task in task_names]
        ax1.plot(task_names, mse_values, 'o-', label=strategy_name, linewidth=2, markersize=8)
    
    ax1.set_xlabel('Task', fontweight='bold')
    ax1.set_ylabel('MSE', fontweight='bold')
    ax1.set_title('Performance Across Tasks', fontweight='bold')
    ax1.legend()
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(True, alpha=0.3)
    ax1.set_yscale('log')
    
    # Plot 2-6: Prediction quality for each task
    task_display_names = ['Efficiency', 'Power Loss', 'Thermal Rise', 'Torque Ripple', 'Vibration']
    colors = ['b', 'r', 'g', 'm', 'c']
    
    plot_idx = 1
    for i, (task_name, display_name, color) in enumerate(zip(task_names, task_display_names, colors)):
        if plot_idx >= 6:  # We have 6 subplots available
            break
            
        ax = axes[0, plot_idx] if plot_idx <= 2 else axes[1, plot_idx - 3]
        
        for strategy_name, result in results.items():
            predictions = result[task_name]['predictions'].flatten()
            actuals = y_tasks[task_name].flatten()
            
            ax.scatter(actuals, predictions, alpha=0.6, label=strategy_name, s=15)
        
        # Perfect prediction line
        min_val = min(np.min(actuals), np.min([results[s][task_name]['predictions'].min() 
                                              for s in results.keys()]))
        max_val = max(np.max(actuals), np.max([results[s][task_name]['predictions'].max() 
                                              for s in results.keys()]))
        ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, label='Perfect')
        
        ax.set_xlabel(f'Actual {display_name}', fontweight='bold')
        ax.set_ylabel(f'Predicted {display_name}', fontweight='bold')
        ax.set_title(f'{display_name} Prediction', fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        
        plot_idx += 1
    
    plt.tight_layout()
    plt.show()
    
    # Print results summary
    print("\n📊 Multi-Task Learning Results:")
    print("=" * 50)
    
    for strategy_name, strategy_results in results.items():
        print(f"\n🎯 {strategy_name}:")
        for task_name in task_names:
            mse = strategy_results[task_name]['mse']
            print(f"  {task_name}: MSE = {mse:.4f}")
    
    # Calculate average improvement
    if 'Single Task (Efficiency)' in results and 'Shared Multi-Task' in results:
        single_avg = np.mean([results['Single Task (Efficiency)'][task]['mse'] 
                             for task in task_names])
        shared_avg = np.mean([results['Shared Multi-Task'][task]['mse'] 
                             for task in task_names])
        improvement = (1 - shared_avg / single_avg) * 100
        print(f"\n🚀 Multi-Task vs Single Task: {improvement:.1f}% average improvement")
    
    return results

# Run multi-task learning demonstration
multitask_results = demonstrate_multi_task_learning()

## Chapter Summary

### Key Takeaways

1. **Internal Transfer Learning**: Transferring knowledge within the same motor type across different operating conditions or power ratings can significantly improve performance with limited target data.

2. **External Transfer Learning**: Domain adaptation techniques like Domain Adversarial Neural Networks (DANN) enable effective knowledge transfer from simulation to real-world data despite domain shifts.

3. **Multi-Task Learning**: Simultaneous learning of related motor performance tasks through shared representations improves data efficiency and generalization.

4. **Practical Considerations**: Successful transfer learning requires careful consideration of domain similarity, data availability, and appropriate transfer strategies.

### Transfer Learning Strategies

- **Fine-tuning**: Adapting pretrained models to target domains with limited data
- **Feature Extraction**: Using pretrained models as fixed feature extractors
- **Domain Adaptation**: Reducing domain discrepancy through adversarial training
- **Multi-Task Learning**: Leveraging shared representations across related tasks

### Next Steps

In the next chapter, we will explore uncertainty quantification techniques for motor performance prediction, enabling reliable decision-making under uncertain conditions.